Em qual horário houve mais avistamentos verdadeiros?

In [0]:
%sql
SELECT hora, COUNT(*) AS avistamentos_verdadeiros
FROM workspace.gold.fato_avistamentos
WHERE is_verified = 1
GROUP BY hora
ORDER BY avistamentos_verdadeiros DESC

hora,avistamentos_verdadeiros
21,5308
20,5109
22,4843
19,4632
18,4111
23,3703
17,3463
16,3046
0,2702
15,2394


Qual é a zona com mais crimes e que é menos patrulha?

In [0]:
%sql
WITH crimes_zona AS (
    SELECT zone_id, COUNT(*) AS crimes
    FROM workspace.gold.fato_crimes
    GROUP BY zone_id
),
base AS (
    SELECT l.borough, l.district, c.crimes, l.patrol_activity_score
    FROM workspace.gold.dim_local l
    JOIN crimes_zona c ON c.zone_id = l.zone_id
),
medianas AS (
    SELECT percentile_approx(crimes, 0.5) AS mediana_crimes,
    percentile_approx(patrol_activity_score, 0.5) AS mediana_patrulha
    FROM base
)
SELECT b.borough, b.district, b.crimes, b.patrol_activity_score
FROM base B CROSS JOIN medianas m
WHERE b.crimes > m.mediana_crimes
AND b.patrol_activity_score < m.mediana_patrulha
ORDER BY b.crimes DESC

borough,district,crimes,patrol_activity_score
Manhattan,Manhattan_South_5,2894,0.284
Staten_Island,Staten_Island_Central_0,1880,0.271
Staten_Island,Staten_Island_East_3,1836,0.119
Bronx,Bronx_Southeast_4,1447,0.077
Manhattan,Manhattan_North_1,1383,0.279
Bronx,Bronx_South_5,1202,0.155
Brooklyn,Brooklyn_North_10,1201,0.222
Queens,Queens_South_5,1183,0.24
Queens,Queens_Southwest_6,1151,0.271
Bronx,Bronx_North_1,1131,0.285


Qual é o lugar mais frequentado pelo Aranha?

In [0]:
%sql
SELECT l.borough, l.district, COUNT(*) AS avistamentos_verdadeiros
FROM workspace.gold.fato_avistamentos a
JOIN workspace.gold.dim_local l ON a.zone_id = l.zone_id
WHERE a.is_verified = 1
GROUP BY l.borough, l.district
ORDER BY avistamentos_verdadeiros DESC
LIMIT 10

borough,district,avistamentos_verdadeiros
Manhattan,Manhattan_Northwest_8,1918
Manhattan,Manhattan_West_7,1870
Manhattan,Manhattan_South_5,1759
Manhattan,Manhattan_North_1,1683
Brooklyn,Brooklyn_Central_0,1656
Manhattan,Manhattan_Northeast_2,1648
Manhattan,Manhattan_East_3,1634
Brooklyn,Brooklyn_North_1,1571
Brooklyn,Brooklyn_Northeast_2,1334
Brooklyn,Brooklyn_Southeast_4,1329


Qual é a porcentagem de vezes que a polícia foi acionada de manhã?

In [0]:
%sql
SELECT COUNT(*) AS crimes_manha,
SUM(CASE WHEN police_called THEN 1 ELSE 0 END) AS com_policia_acionada,
ROUND(AVG(CASE WHEN police_called THEN 1 ELSE 0 END) * 100, 1) AS pct_policia_acionada
FROM workspace.gold.fato_crimes
WHERE time_of_day = 'morning'

crimes_manha,com_policia_acionada,pct_policia_acionada
7569,6158,81.4
